# Artist deep dive: John Scofield

Real data, not a fixture -- same real ingested dataset as
`02-explore-real-discogs-catalog.ipynb` (needs the same `SNAPSHOT` env var
and a completed `scripts/run-ingest.sh` run).

**Evidence-based identity, not a name match.** A credited name string
("John Scofield") is not the same thing as a single real person -- Discogs
disambiguates via a numeric `artist_id` (PAN identity), separate from the
credited display text (ANV). This notebook discovers every distinct
`artist_id` credited under names matching "Scofield" first, and picks the
real jazz guitarist's identity by evidence (overwhelming credit count), not
by assuming the first name match is correct -- the same methodology
`packages/catalog` itself is built around. See `docs/DISCOGS_INGESTION.md`'s
PAN/ANV section.


In [ ]:
import os
from pathlib import Path

PROCESSED_ROOT = Path("/data/discogs")
snapshot = os.environ.get("SNAPSHOT")
if not snapshot:
    raise RuntimeError(
        "Set SNAPSHOT (e.g. SNAPSHOT=20260601) as a Jupyter kernel/container "
        "env var, matching a completed scripts/run-ingest.sh run. See "
        "docs/DISCOGS_INGESTION.md."
    )

dataset_root = PROCESSED_ROOT / f"snapshot={snapshot}"
releases_glob = str(dataset_root / "table=releases" / "*.parquet")
credits_glob = str(dataset_root / "table=credits" / "*.parquet")

if not (dataset_root / "table=releases").is_dir():
    raise RuntimeError(
        f"No dataset at {dataset_root} -- confirm DISCOGS_PROCESSED_DIR in "
        "local/dask.env points at the right host path, and that "
        f"snapshot={snapshot} has actually completed (docs/DISCOGS_INGESTION.md)."
    )
print(f"Using real dataset at {dataset_root}")

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE VIEW releases AS SELECT * FROM '{releases_glob}'")
con.execute(f"CREATE VIEW credits AS SELECT * FROM '{credits_glob}'")

In [ ]:
# Discover every distinct linked identity credited under a name matching
# "scofield" -- real evidence before committing to one artist_id. Multiple
# unrelated people can share a surname; do not assume the first row is
# right.
candidates = con.sql(
    """
    SELECT artist_id, name, count(*) n
    FROM credits
    WHERE name ILIKE '%scofield%' AND is_linked
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
    """
).df()
print("Candidate identities credited under a name matching 'scofield':")
print(candidates.to_string(index=False))

ARTIST_ID = int(candidates.iloc[0]["artist_id"])
print(
    f"\nUsing artist_id={ARTIST_ID} ({candidates.iloc[0]['name']!r}, "
    f"{candidates.iloc[0]['n']} credits) -- the dominant real identity by "
    "far, not merely the first alphabetical or first-seen match."
)

In [ ]:
# Minimal, dependency-free inline-SVG bar chart -- the Jupyter image
# deliberately doesn't include matplotlib/plotly (see infra/dask/requirements.txt);
# reuses the same validated categorical palette as local/benchmarks/report.html
# (see the dataviz skill's references/palette.md for the source values).
import html as _html
import math

from IPython.display import HTML

_PALETTE = ["#2a78d6", "#1baf7a", "#eda100", "#e34948"]


def bar_chart(data, title, color=_PALETTE[0], value_fmt="{:,}"):
    """data: list of (label, value) tuples, already sorted as desired."""
    if not data:
        return HTML(f"<p><em>{_html.escape(title)}: no data.</em></p>")
    max_val = max(v for _, v in data)
    magnitude = 10 ** math.floor(math.log10(max_val)) if max_val > 0 else 1
    axis_max = math.ceil(max_val / magnitude) * magnitude if max_val > 0 else 1

    bar_w, gap, chart_h, top_pad = 24, 14, 180, 24
    width = max(420, len(data) * (bar_w + gap) + 70)

    bars = []
    for i, (label, value) in enumerate(data):
        x = 55 + i * (bar_w + gap)
        h = (value / axis_max) * chart_h if axis_max else 0
        y = top_pad + chart_h - h
        bars.append(
            f'<rect x="{x}" y="{y:.1f}" width="{bar_w}" height="{h:.1f}" rx="4" fill="{color}"/>'
        )
        bars.append(
            f'<text x="{x + bar_w / 2}" y="{y - 6:.1f}" font-size="11" '
            f'text-anchor="middle" fill="#0b0b0b">{value_fmt.format(value)}</text>'
        )
        label_str = _html.escape(str(label))[:16]
        bars.append(
            f'<text x="{x + bar_w / 2}" y="{top_pad + chart_h + 18}" font-size="10" '
            f'text-anchor="middle" fill="#898781">{label_str}</text>'
        )

    grid = []
    for frac in (0, 0.5, 1.0):
        gy = top_pad + chart_h - frac * chart_h
        grid.append(
            f'<line x1="50" y1="{gy:.1f}" x2="{width - 10}" y2="{gy:.1f}" '
            f'stroke="#e1e0d9" stroke-width="1"/>'
        )
        grid.append(
            f'<text x="45" y="{gy + 4:.1f}" font-size="10" text-anchor="end" '
            f'fill="#898781">{int(axis_max * frac):,}</text>'
        )

    svg_style = (
        "font-family: system-ui, -apple-system, 'Segoe UI', sans-serif; margin-bottom: 12px;"
    )
    title_style = "font-size:14px; font-weight:600; margin-bottom:6px; color:#0b0b0b;"
    svg = f"""
    <div style="{svg_style}">
      <div style="{title_style}">{_html.escape(title)}</div>
      <svg viewBox="0 0 {width} {top_pad + chart_h + 40}" width="100%"
           height="{top_pad + chart_h + 40}">
        {"".join(grid)}
        {"".join(bars)}
      </svg>
    </div>
    """
    return HTML(svg)

In [ ]:
role_rows = con.execute(
    "SELECT coalesce(role_text, '(none)') role_text, count(*) n "
    "FROM credits WHERE artist_id = ? GROUP BY 1 ORDER BY 2 DESC LIMIT 8",
    [ARTIST_ID],
).fetchall()
bar_chart(role_rows, "Credits by role (top 8)", color=_PALETTE[0])

In [ ]:
decade_rows = con.execute(
    """
    SELECT (CAST(substr(r.released, 1, 4) AS INT) // 10) * 10 AS decade,
           count(DISTINCT r.release_id) n
    FROM releases r JOIN credits c USING (release_id)
    WHERE c.artist_id = ? AND r.released ~ '^[0-9]{4}'
    GROUP BY 1 ORDER BY 1
    """,
    [ARTIST_ID],
).fetchall()
decade_rows = [(f"{d}s", n) for d, n in decade_rows]
bar_chart(decade_rows, "Releases by decade", color=_PALETTE[1])

In [ ]:
country_rows = con.execute(
    """
    SELECT coalesce(r.country, '(unknown)') country, count(DISTINCT r.release_id) n
    FROM releases r JOIN credits c USING (release_id)
    WHERE c.artist_id = ?
    GROUP BY 1 ORDER BY 2 DESC LIMIT 8
    """,
    [ARTIST_ID],
).fetchall()
bar_chart(country_rows, "Release country (top 8)", color=_PALETTE[2])

In [ ]:
# Co-credited, linked artists on the same release -- evidence of shared
# billing, not a claim about musical influence or personal relationship
# (see AGENTS.md: "Do not infer artistic influence, relationships, or
# intent from a shared credit").
collab_rows = con.execute(
    """
    SELECT c2.name, count(DISTINCT c2.release_id) n
    FROM credits c1 JOIN credits c2 USING (release_id)
    WHERE c1.artist_id = ? AND c2.artist_id != ? AND c2.is_linked
    GROUP BY 1 ORDER BY 2 DESC LIMIT 8
    """,
    [ARTIST_ID, ARTIST_ID],
).fetchall()
bar_chart(collab_rows, "Most frequent co-credited artists (shared releases)", color=_PALETTE[3])

In [ ]:
# Distributed cross-check: scan the FULL, un-narrowed credits table (hundreds
# of millions of rows across the whole catalog) with Dask, not just this
# artist's already-small slice -- confirms the real worker cluster, not
# DuckDB alone, agrees on the total credit-row count for this artist_id.
import dask.dataframe as dd
from dask.distributed import Client

client = Client("tcp://dask-scheduler:8786")
credits_dd = dd.read_parquet(credits_glob, columns=["artist_id", "release_id"])
dask_count = int((credits_dd["artist_id"] == ARTIST_ID).sum().compute())
duckdb_count = con.execute(
    "SELECT count(*) FROM credits WHERE artist_id = ?", [ARTIST_ID]
).fetchone()[0]
assert dask_count == duckdb_count, (dask_count, duckdb_count)
print(
    f"Total credit rows for artist_id={ARTIST_ID}: {dask_count:,} "
    "(Dask distributed scan, cross-checked against DuckDB)"
)

In [ ]:
# Sample analysis -- generated from the live query results above, not
# hardcoded prose, so it stays accurate as the underlying snapshot changes.
total_releases = con.execute(
    "SELECT count(DISTINCT release_id) FROM credits WHERE artist_id = ?", [ARTIST_ID]
).fetchone()[0]
year_range = con.execute(
    """
    SELECT min(CAST(substr(r.released,1,4) AS INT)),
           max(CAST(substr(r.released,1,4) AS INT))
    FROM releases r JOIN credits c USING (release_id)
    WHERE c.artist_id = ? AND r.released ~ '^[0-9]{4}'
    """,
    [ARTIST_ID],
).fetchone()
top_role, top_role_n = role_rows[0]
top_collab, top_collab_n = collab_rows[0]
top_decade, top_decade_n = max(decade_rows, key=lambda row: row[1])

print(f"snapshot={snapshot}")
print(f"John Scofield (artist_id={ARTIST_ID}) appears on {total_releases:,} distinct releases,")
print(
    f"spanning {year_range[0]}-{year_range[1]}, peaking in the {top_decade} "
    f"({top_decade_n} releases)."
)
print(f"Most common credited role: {top_role!r} ({top_role_n} credits).")
print(f"Most frequent co-credited artist: {top_collab} ({top_collab_n} shared releases) --")
print("evidence of shared billing only, not a claim of collaboration frequency or influence.")